# Deep Retention — Limpieza y preprocesamiento

Este notebook carga `Customer-Churn-Records.csv`, limpia una copia, crea
particiones estratificadas 70/15/15 y ajusta el preprocesador exclusivamente
con entrenamiento. La exportación está desactivada por defecto para evitar
sobrescribir artefactos sin una ejecución controlada.


## 1. Configuración y carga


In [1]:
import hashlib
import json
import os
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import sklearn
from IPython.display import display
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

PROJECT_ROOT = next(
    candidate
    for candidate in [Path.cwd(), *Path.cwd().parents]
    if (candidate / "data" / "Customer-Churn-Records.csv").is_file()
)
DATA_PATH = PROJECT_ROOT / "data" / "Customer-Churn-Records.csv"
ARTIFACT_ROOT = PROJECT_ROOT
EXPORT_ARTIFACTS = True

PROCESSED_DIR = ARTIFACT_ROOT / "data" / "processed"
MODELS_DIR = ARTIFACT_ROOT / "models"
RANDOM_STATE = 42
TARGET = "Exited"
FEATURES = [
    "CreditScore", "Geography", "Age", "Tenure", "Balance",
    "NumOfProducts", "HasCrCard", "IsActiveMember", "EstimatedSalary",
]
NUMERIC_FEATURES = [
    "CreditScore", "Age", "Tenure", "Balance",
    "NumOfProducts", "EstimatedSalary",
]
BINARY_FEATURES = ["HasCrCard", "IsActiveMember"]

df_original = pd.read_csv(DATA_PATH)
print(f"Dataset cargado: {df_original.shape[0]:,} × {df_original.shape[1]}")


Dataset cargado: 10,000 × 18


## 2. Limpieza sobre una copia


In [2]:
expected_columns = [
    "RowNumber", "CustomerId", "Surname", "CreditScore", "Geography",
    "Gender", "Age", "Tenure", "Balance", "NumOfProducts", "HasCrCard",
    "IsActiveMember", "EstimatedSalary", "Exited", "Complain",
    "Satisfaction Score", "Card Type", "Point Earned",
]
missing_columns = sorted(set(expected_columns) - set(df_original.columns))
if missing_columns:
    raise ValueError(f"Faltan columnas requeridas: {missing_columns}")

df_clean = df_original.copy()
text_columns = df_clean.select_dtypes(include="object").columns
for column in text_columns:
    df_clean[column] = df_clean[column].astype("string").str.strip()

unknown_surname_mask = df_clean["Surname"].str.contains(
    "?", regex=False, na=False
)
unknown_surnames = int(unknown_surname_mask.sum())
df_clean.loc[unknown_surname_mask, "Surname"] = pd.NA

cleaning_summary = pd.DataFrame(
    {
        "control": [
            "Filas antes", "Filas después", "Apellidos con ? detectados",
            "Surname faltantes resultantes", "Filas duplicadas",
            "CustomerId duplicados",
        ],
        "resultado": [
            len(df_original), len(df_clean), unknown_surnames,
            int(df_clean["Surname"].isna().sum()),
            int(df_clean.duplicated().sum()),
            int(df_clean["CustomerId"].duplicated().sum()),
        ],
    }
)
display(cleaning_summary)


,control,resultado
0,Filas antes,10000
1,Filas después,10000
2,Apellidos con ? detectados,92
3,Surname faltantes resultantes,92
4,Filas duplicadas,0
5,CustomerId duplicados,0


Los 92 apellidos con caracteres desconocidos se representan mediante
`pd.NA`. No se reconstruyen nombres, no se eliminan filas y los 10.000
identificadores permanecen disponibles para trazabilidad. Los saldos cero
se conservan porque son valores válidos.


## 3. Controles esenciales y selección de variables


In [3]:
binary_columns = ["HasCrCard", "IsActiveMember", "Exited", "Complain"]
binary_validity = {
    column: set(df_clean[column].dropna().unique()).issubset({0, 1})
    for column in binary_columns
}
range_validity = {
    "CreditScore": df_clean["CreditScore"].between(300, 850).all(),
    "Age": df_clean["Age"].between(18, 100).all(),
    "Tenure": df_clean["Tenure"].between(0, 10).all(),
    "Balance": df_clean["Balance"].ge(0).all(),
    "NumOfProducts": df_clean["NumOfProducts"].between(1, 4).all(),
    "EstimatedSalary": df_clean["EstimatedSalary"].ge(0).all(),
    "Satisfaction Score": df_clean["Satisfaction Score"].between(1, 5).all(),
    "Point Earned": df_clean["Point Earned"].ge(0).all(),
}
display(pd.Series(binary_validity, name="dominio_binario_valido").to_frame())
display(pd.Series(range_validity, name="rango_valido").to_frame())

X = df_clean[FEATURES].copy()
y = df_clean[TARGET].astype(np.int8).copy()
if X.isna().any().any() or not np.isfinite(
    X[NUMERIC_FEATURES + BINARY_FEATURES].to_numpy(dtype=float)
).all():
    raise ValueError("Las predictoras contienen valores faltantes o no finitos.")
if not all(binary_validity.values()) or not all(range_validity.values()):
    raise ValueError("Existen dominios o rangos inconsistentes.")
if "Complain" in X.columns or TARGET in X.columns or "CustomerId" in X.columns:
    raise ValueError("La matriz predictora contiene una columna excluida.")

print("Predictoras:", FEATURES)
print("Objetivo:", TARGET)
print("Complain excluida por riesgo de fuga de información.")


,dominio_binario_valido
HasCrCard,True
IsActiveMember,True
Exited,True
Complain,True


,rango_valido
CreditScore,True
Age,True
Tenure,True
Balance,True
NumOfProducts,True
EstimatedSalary,True
Satisfaction Score,True
Point Earned,True


Predictoras: ['CreditScore', 'Geography', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'HasCrCard', 'IsActiveMember', 'EstimatedSalary']
Objetivo: Exited
Complain excluida por riesgo de fuga de información.


## 4. Particiones estratificadas 70/15/15


In [4]:
all_indices = df_clean.index.to_numpy()
train_indices, temporary_indices = train_test_split(
    all_indices,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)
validation_indices, test_indices = train_test_split(
    temporary_indices,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y.loc[temporary_indices],
)

split_sets = [set(train_indices), set(validation_indices), set(test_indices)]
overlap = any(
    split_sets[left] & split_sets[right]
    for left in range(3)
    for right in range(left + 1, 3)
)
if overlap or len(set().union(*split_sets)) != len(df_clean):
    raise ValueError("Las particiones se solapan o no cubren el dataset.")

split_summary = pd.DataFrame(
    {
        "partición": ["train", "validation", "test"],
        "filas": [len(train_indices), len(validation_indices), len(test_indices)],
        "porcentaje": [
            len(train_indices) / len(df_clean) * 100,
            len(validation_indices) / len(df_clean) * 100,
            len(test_indices) / len(df_clean) * 100,
        ],
        "Exited (%)": [
            y.loc[train_indices].mean() * 100,
            y.loc[validation_indices].mean() * 100,
            y.loc[test_indices].mean() * 100,
        ],
    }
)
display(split_summary)


,partición,filas,porcentaje,Exited (%)
0,train,7000,70.0,20.385714
1,validation,1500,15.0,20.400000
2,test,1500,15.0,20.333333


El conjunto de prueba queda reservado para la evaluación final. La
estratificación conserva prácticamente la misma proporción de abandono en
las tres particiones y `random_state=42` permite reproducir la división.


## 5. Ajuste del preprocesador solo con entrenamiento


In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), NUMERIC_FEATURES),
        (
            "geography",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
            ["Geography"],
        ),
        ("binary", "passthrough", BINARY_FEATURES),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

X_train = X.loc[train_indices]
X_validation = X.loc[validation_indices]
X_test = X.loc[test_indices]
y_train = y.loc[train_indices].to_numpy(dtype=np.int8)
y_validation = y.loc[validation_indices].to_numpy(dtype=np.int8)
y_test = y.loc[test_indices].to_numpy(dtype=np.int8)

X_train_processed = np.asarray(
    preprocessor.fit_transform(X_train), dtype=np.float32
)
X_validation_processed = np.asarray(
    preprocessor.transform(X_validation), dtype=np.float32
)
X_test_processed = np.asarray(
    preprocessor.transform(X_test), dtype=np.float32
)
transformed_features = preprocessor.get_feature_names_out().tolist()

dimensions = pd.DataFrame(
    {
        "partición": ["train", "validation", "test"],
        "X": [
            X_train_processed.shape,
            X_validation_processed.shape,
            X_test_processed.shape,
        ],
        "y": [y_train.shape, y_validation.shape, y_test.shape],
    }
)
display(dimensions)
print("Características transformadas:", transformed_features)


,partición,X,y
0,train,"(7000, 11)","(7000,)"
1,validation,"(1500, 11)","(1500,)"
2,test,"(1500, 11)","(1500,)"


Características transformadas: ['CreditScore', 'Age', 'Tenure', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'Geography_France', 'Geography_Germany', 'Geography_Spain', 'HasCrCard', 'IsActiveMember']


In [6]:
processed_arrays = [
    X_train_processed, X_validation_processed, X_test_processed
]
finite_checks = [bool(np.isfinite(array).all()) for array in processed_arrays]
scaler = preprocessor.named_transformers_["numeric"]
scaler_check = pd.DataFrame(
    {
        "variable": NUMERIC_FEATURES,
        "media_entrenamiento": X_train[NUMERIC_FEATURES].mean().to_numpy(),
        "media_aprendida_scaler": scaler.mean_,
    }
)
display(scaler_check)
print("Matrices sin NaN ni infinitos:", all(finite_checks))
if not all(finite_checks):
    raise ValueError("El preprocesamiento produjo valores no finitos.")


,variable,media_entrenamiento,media_aprendida_scaler
0,CreditScore,650.781000,650.781000
1,Age,38.945429,38.945429
2,Tenure,5.004000,5.004000
3,Balance,76656.243549,76656.243549
4,NumOfProducts,1.531000,1.531000
5,EstimatedSalary,99488.809789,99488.809789


Matrices sin NaN ni infinitos: True


## 6. Exportación controlada y contrato de artefactos


In [7]:
source_sha256 = hashlib.sha256(DATA_PATH.read_bytes()).hexdigest()
artifact_payload = json.dumps(
    {
        "source_sha256": source_sha256,
        "random_state": RANDOM_STATE,
        "features": FEATURES,
        "train_ids": sorted(
            df_clean.loc[train_indices, "CustomerId"].astype(int).tolist()
        ),
    },
    separators=(",", ":"),
).encode()
artifact_id = hashlib.sha256(artifact_payload).hexdigest()[:16]
preprocessor.artifact_id_ = artifact_id

metadata = {
    "preprocessing_schema_version": "2.0",
    "artifact_id": artifact_id,
    "source": {
        "filename": DATA_PATH.name,
        "sha256": source_sha256,
        "records": int(len(df_clean)),
        "columns": int(df_clean.shape[1]),
    },
    "features": FEATURES,
    "excluded_for_leakage": ["Complain"],
    "transformed_features": transformed_features,
    "split": {
        "random_state": RANDOM_STATE,
        "train": 0.70,
        "validation": 0.15,
        "test": 0.15,
    },
    "datasets": {
        "train": {"file": "train.npz", "X_shape": list(X_train_processed.shape)},
        "validation": {"file": "validation.npz", "X_shape": list(X_validation_processed.shape)},
        "test": {"file": "test.npz", "X_shape": list(X_test_processed.shape)},
    },
    "scikit_learn_version": sklearn.__version__,
    "preprocessor_file": "preprocesador.joblib",
    "clean_dataset_file": "clientes_limpios.csv",
    "unknown_surnames": unknown_surnames,
}

if EXPORT_ARTIFACTS:
    PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
    MODELS_DIR.mkdir(parents=True, exist_ok=True)
    df_clean.to_csv(PROCESSED_DIR / "clientes_limpios.csv", index=False)
    joblib.dump(preprocessor, MODELS_DIR / "preprocesador.joblib")

    partitions = {
        "train": (X_train_processed, y_train, train_indices),
        "validation": (
            X_validation_processed, y_validation, validation_indices
        ),
        "test": (X_test_processed, y_test, test_indices),
    }
    for name, (features, labels, indices) in partitions.items():
        np.savez_compressed(
            PROCESSED_DIR / f"{name}.npz",
            X=features,
            y=labels,
            row_indices=np.asarray(indices, dtype=np.int64),
            customer_ids=df_clean.loc[indices, "CustomerId"].to_numpy(dtype=np.int64),
            artifact_id=np.asarray(artifact_id),
        )
    with (PROCESSED_DIR / "metadata.json").open("w", encoding="utf-8") as file:
        json.dump(metadata, file, ensure_ascii=False, indent=2)
    print("Exportación controlada completada.")
else:
    print("Exportación desactivada: no se modificaron artefactos versionados.")


Exportación controlada completada.


## 7. Verificación de la exportación


In [8]:
if EXPORT_ARTIFACTS:
    reloaded_preprocessor = joblib.load(MODELS_DIR / "preprocesador.joblib")
    with np.load(PROCESSED_DIR / "train.npz", allow_pickle=False) as saved:
        saved_train_shape = saved["X"].shape
        saved_artifact_id = str(saved["artifact_id"].item())
    with (PROCESSED_DIR / "metadata.json").open(encoding="utf-8") as file:
        saved_metadata = json.load(file)

    export_checks = pd.Series(
        {
            "train_shape": saved_train_shape == X_train_processed.shape,
            "artifact_id_npz": saved_artifact_id == artifact_id,
            "artifact_id_metadata": saved_metadata["artifact_id"] == artifact_id,
            "artifact_id_preprocessor": getattr(
                reloaded_preprocessor, "artifact_id_", None
            ) == artifact_id,
            "feature_names": reloaded_preprocessor.get_feature_names_out().tolist()
            == transformed_features,
        },
        name="resultado",
    )
    display(export_checks.to_frame())
    if not export_checks.all():
        raise ValueError("La exportación no superó todos los controles.")
else:
    print("La verificación de archivos se ejecutará al habilitar la exportación.")


,resultado
train_shape,True
artifact_id_npz,True
artifact_id_metadata,True
artifact_id_preprocessor,True
feature_names,True


## 8. Resultado

El preprocesador utiliza seis variables escaladas, tres categorías de país
codificadas y dos variables binarias conservadas, para un total de 11
características transformadas. `Complain`, `Exited`, identificadores y
columnas adicionales quedan fuera de `X`. Para regenerar los artefactos se
debe ejecutar de forma controlada con
`DEEP_RETENTION_EXPORT_ARTIFACTS=1`; el notebook 03 consumirá exactamente
ese mismo `artifact_id`.
